In [2]:
# ==========================================
# 1️⃣ Imports
# ==========================================
import pandas as pd
import numpy as np
import mlflow
import mlflow.lightgbm

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.ensemble import IsolationForest
from lightgbm import LGBMClassifier


# ==========================================
# 2️⃣ Memory-Safe Data Loader
# ==========================================
def load_data_efficient(path, chunksize=100000):
    chunks = []
    
    for chunk in pd.read_csv(path, chunksize=chunksize):
        
        # Downcast floats
        for col in chunk.select_dtypes(include=['float64']).columns:
            chunk[col] = chunk[col].astype('float32')
        
        # Downcast ints
        for col in chunk.select_dtypes(include=['int64']).columns:
            chunk[col] = chunk[col].astype('int32')
            
        chunks.append(chunk)
    
    return pd.concat(chunks, axis=0)


print("📥 Loading transaction data...")
txn = load_data_efficient('../data/raw/train_transaction.csv')

print("📥 Loading identity data...")
idn = load_data_efficient('../data/raw/train_identity.csv')

print("🔗 Merging datasets...")
df = txn.merge(idn, on='TransactionID', how='left')

print("✅ Data loaded successfully")
print(f"Fraud Rate: {df['isFraud'].mean():.3%}")


# ==========================================
# 3️⃣ Feature Selection (Numeric Only)
# ==========================================
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c not in ['TransactionID', 'isFraud']]

X = df[num_cols].fillna(-999)
y = df['isFraud']


# ==========================================
# 4️⃣ Train/Test Split
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


# ==========================================
# 5️⃣ Add Anomaly Score Feature
# ==========================================
print("🧠 Training IsolationForest...")

iso = IsolationForest(
    n_estimators=100,
    contamination=0.035,
    random_state=42,
    n_jobs=-1
)

X_train["anomaly_score"] = iso.fit_predict(X_train)
X_test["anomaly_score"] = iso.predict(X_test)


# ==========================================
# 6️⃣ Train LightGBM (No SMOTE — safer)
# ==========================================
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

mlflow.set_tracking_uri("http://localhost:5000")

with mlflow.start_run(run_name="fraud_lgbm_memory_safe"):
    
    model = LGBMClassifier(
        n_estimators=400,
        learning_rate=0.05,
        num_leaves=64,
        max_depth=-1,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1
    )
    
    model.fit(X_train, y_train)
    
    preds = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, preds)
    
    mlflow.log_metric("fraud_auc", auc)
    mlflow.lightgbm.log_model(
        model,
        artifact_path="fraud-model",
        registered_model_name="fraud-detection-lgbm"
    )
    
    print(f"\n🚀 Fraud AUC: {auc:.4f}\n")
    print(classification_report(y_test, (preds > 0.5).astype(int)))

print("✅ Training complete and model registered.")

c:\Users\Shanu\Desktop\Final_projects\fraud_detection\credit-risk-platform\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📥 Loading transaction data...
📥 Loading identity data...
🔗 Merging datasets...
✅ Data loaded successfully
Fraud Rate: 3.499%
🧠 Training IsolationForest...
[LightGBM] [Info] Number of positive: 16530, number of negative: 455902
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.877892 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 38365
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 401
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.034989 -> initscore=-3.317101
[LightGBM] [Info] Start training from score -3.317101


2026/03/05 21:08:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/05 21:08:35 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'fraud-detection-lgbm' already exists. Creating a new version of this model...
2026/03/05 21:09:12 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: fraud-detection-lgbm, version 2
Created version '2' of model 'fraud-detection-lgbm'.



🚀 Fraud AUC: 0.9536

              precision    recall  f1-score   support

           0       0.99      0.93      0.96    113975
           1       0.31      0.84      0.45      4133

    accuracy                           0.93    118108
   macro avg       0.65      0.89      0.71    118108
weighted avg       0.97      0.93      0.94    118108

🏃 View run fraud_lgbm_memory_safe at: http://localhost:5000/#/experiments/0/runs/aec34a3418274b7593597f0183811da3
🧪 View experiment at: http://localhost:5000/#/experiments/0
✅ Training complete and model registered.
